# 06. Code Interpreter -- Model-Written, Server-Executed Python

This notebook (**AI-103 -> Section 01**) attaches the built-in `code_interpreter` tool so the model can write and *actually execute* Python in a sandbox to answer a precise numeric question (compound interest), then inspects the structured `response.output` list to separate the generated code, its execution results, and the final natural-language answer.

**Difficulty: Intermediate/Advanced**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `openai`, `azure-identity`, `python-dotenv`

**Azure resources**
- An Azure OpenAI / AI Foundry deployment with the built-in `code_interpreter` tool available.
- Entra ID auth via `az login`.

**Environment variables** -- add to the root `.env`:
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```


## What You'll Learn
- The built-in `code_interpreter` tool and the `container: {"type": "auto"}` sandbox lifecycle
- Why executing code beats asking an LLM to do exact arithmetic in its head
- `response.output` as a list of typed items, vs. the `response.output_text` shortcut
- Discriminating `code_interpreter_call` items from `message` items by `item.type`


### 1. Imports and configuration


In [ ]:
import os

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")


### 2. Build the client


In [ ]:
client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
)


### 3. Ask a precise numeric question with the code interpreter attached

`tools=[{"type": "code_interpreter", "container": {"type": "auto"}}]` lets the model write Python and run it in a real sandbox rather than estimating the answer purely from its language modeling. `container: {"type": "auto"}` means the service fully manages the sandbox's lifecycle (provisioning and teardown) -- you don't provide or manage any compute yourself.

**Exam tip:** LLMs are unreliable at exact multi-step arithmetic; the Code Interpreter tool runs generated Python inside an isolated, ephemeral, service-managed container, giving deterministic, verifiable numeric results. This is well suited for precise calculation, data analysis, and chart generation -- an AI-102 exam scenario describing "let the model write and run code to compute an exact answer" is pointing at this tool.


In [ ]:
response = client.responses.create(
    model=deployment_name,
    instructions="You are a data analyst. Use Python to calculate precisely.",
    input="What is the compound interest on $10,000 at 5 percent annual rate over 10 years?",
    tools=[{"type": "code_interpreter", "container": {"type": "auto"}}],
)


### 4. Inspect what happened under the hood

`response.output` is a list of typed items rather than a single message. We loop through it: `code_interpreter_call` items expose `item.code` (the Python the model wrote) and `item.outputs` (what running it produced); `message` items are the normal assistant text, which we still read via the `response.output_text` convenience property.

**Exam tip:** `response.output_text` only concatenates `message`-type text. If you need the full audit trail of what tools were invoked, with what code and what results -- important for debugging and for compliance in enterprise scenarios -- iterate `response.output` directly and branch on `item.type`. This "structured output items" model is a defining shape of the Responses API that AI-102 associates with its agentic-tool-use pattern.

**Alternatives:** For a persistent, multi-turn agent that keeps a code interpreter attached across many turns (rather than one-off stateless calls like this), use the Azure AI Agent Service (`azure-ai-agents`) with a `CodeInterpreterTool` on a persistent Agent + Thread -- see `11_azure_ai_foundry/02_tools/code_interpreter_tool.py` in this repo for that equivalent.


In [ ]:
# Inspect what happened under the hood
for item in response.output:
    if item.type == "code_interpreter_call":
        print("=== Python Code the Model Wrote ===")
        print(item.code)
        print("\n=== Output from Execution ===")
        print(item.outputs)
    elif item.type == "message":
        print("\n=== Final Answer ===")
        print(response.output_text)


## Summary

You attached the built-in `code_interpreter` tool, let the model write and execute Python to compute compound interest exactly, and inspected `response.output` to separately print the generated code, its execution output, and the final natural-language answer.


## Try It Yourself
1. **Easy:** Change the principal, rate, or number of years in the question.
2. **Medium:** Ask a question that requires a chart (e.g. "plot the balance year over year") and inspect what new `item.type` values appear in `response.output` for image/file outputs.
3. **Advanced:** Wrap this pattern in a helper function that returns a dict of `{"code": ..., "execution_output": ..., "final_answer": ...}` for logging/auditing every call your app makes.
